<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03c_visualizations_net_change.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing

In [140]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import plotly.express as px
import seaborn as sns


from google.colab import drive
drive.mount('/content/drive')

# !wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
# import functions as fx

import importlib
importlib.reload(fx)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<module 'functions' from '/content/functions.py'>

##Reading in file

In [122]:
#gdf of opening dates after 2016 concatenated with gdf of closing dates after 2016
gdf_open_close = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/open_close_after_2016.geojson")

#all businesses with no end date or end dates after 2016
gdf_biz = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/biz_all_startdate.geojson")

#immediately clipping to exclude openings in 2026
gdf_biz = gdf_biz[(gdf_biz.year_open <=2025)]

##Reading in census tracts and block groups

In [123]:
sf_tracts = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts.geojson')

sf_block_grp = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_block_grp.geojson')

##Joining census tract / grp geometry and grouping by the tract and the year
- pivoting table to create columns of num openings and num closings

In [124]:
open_close_tracts_gdf = fx.group_points_by_poly_year(gdf_open_close, sf_tracts)

open_close_grp_gdf = fx.group_points_by_poly_year(gdf_open_close, sf_block_grp)


##Calculating new metrics


In [125]:
open_close_tracts_gdf = fx.calc_business_dynamics(
    open_close_tracts_gdf, gdf_biz, sf_tracts
)

open_close_grp_gdf = fx.calc_business_dynamics(
    open_close_grp_gdf, gdf_biz, sf_block_grp
)

#Choropleth maps

###getting epc tracts and adding is_epc boolean

In [126]:
epc_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/epc_tracts_sf.geojson")
epc_tracts = epc_tracts.rename(columns={'tract_geoid': 'GEOID'})

open_close_tracts_gdf['is_epc'] = open_close_tracts_gdf['GEOID'].isin(epc_tracts['GEOID'])

###Net entry rate map (net change for year / total businesses active that year)

In [137]:
#commenting out for github

# #block grp
# fx.choropleth_animated(
#     open_close_grp_gdf, 'net_entry_rate', epc_tracts
# )

# #tract level
# fx.choropleth_animated(
#     open_close_tracts_gdf, 'net_entry_rate', epc_tracts
# )


###Map of percent change in businesses compared to 2016 (net change each year / baseline net change in 2016)

In [136]:
# #block group
# fx.choropleth_animated(
#     open_close_grp_gdf, 'growth_pct_over_2016', epc_tracts, 2017
# )

# #tract level
# fx.choropleth_animated(
#     open_close_tracts_gdf, 'growth_pct_over_2016', epc_tracts, 2017
# )

In [131]:
open_close_grp_gdf.columns

Index(['GEOID', 'geometry', 'year', 'closed', 'opened', 'net_change',
       'baseline_2016', 'growth_pct_over_2016', 'biz_stock', 'net_entry_rate',
       'gross_exit_rate'],
      dtype='object')

#Graphs - line graphs

In [139]:
sns.lineopen_close_tracts_gdf

,GEOID,geometry,year,closed,opened,net_change,baseline_2016,growth_pct_over_2016,biz_stock,net_entry_rate,gross_exit_rate,is_epc
0,06075060400,"POLYGON ((-122.5068 37.73556, -122.50614 37.73...",2016.0,0.0,9.0,9.0,9.0,100.000000,42,21.428571,21.428571,False
1,06075060400,"POLYGON ((-122.5068 37.73556, -122.50614 37.73...",2017.0,2.0,10.0,8.0,9.0,88.888889,52,15.384615,19.230769,False
2,06075060400,"POLYGON ((-122.5068 37.73556, -122.50614 37.73...",2018.0,9.0,10.0,1.0,9.0,11.111111,60,1.666667,16.666667,False
3,06075060400,"POLYGON ((-122.5068 37.73556, -122.50614 37.73...",2019.0,6.0,18.0,12.0,9.0,133.333333,69,17.391304,26.086957,False
4,06075060400,"POLYGON ((-122.5068 37.73556, -122.50614 37.73...",2020.0,4.0,1.0,-3.0,9.0,-33.333333,64,-4.687500,1.562500,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2391,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2021.0,38.0,44.0,6.0,61.0,9.836066,462,1.298701,9.523810,True
2392,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2022.0,49.0,33.0,-16.0,61.0,-26.229508,457,-3.501094,7.221007,True
2393,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2023.0,48.0,54.0,6.0,61.0,9.836066,462,1.298701,11.688312,True
2394,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2024.0,43.0,40.0,-3.0,61.0,-4.918033,454,-0.660793,8.810573,True
